<a href="https://colab.research.google.com/github/navikram03/data-cleaning/blob/main/test_data_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BRONZE

In [89]:
%pip install pyspark

In [90]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

df = spark.read.csv("/content/test.csv", inferSchema=True, header=True)

SILVER

In [91]:
df_dict_names = {
    "Rank":"rank",
    "Peak":"peak",
    "All Time Peak":"all_time_peak",
    "Actual gross":"actual_gross",
    "Adjusted gross (in 2022 dollars)":"adjusted_gross",
    "Artist":"artist",
    "Tour title":"tour_title",
    "Year(s)":"year",
    "Shows":"shows",
    "Average gross":"average_gross",
    "Ref.":"reference"
}

df = df.toDF(*[column.replace('\xa0', ' ').strip() for column in df.columns])

for each in df.columns:
  new_name = df_dict_names[each]

  df = df.withColumnRenamed(each, new_name)
  if new_name in ["peak","all_time_peak"]:
    df = df.withColumn(new_name, (regexp_replace(col(new_name),r'\[.*\]','' )).cast("int"))
  elif new_name in ["actual_gross", "adjusted_gross", "average_gross"]:
    df = df.withColumn(new_name, (regexp_replace(col(new_name),r'\[.*\]','' ))) \
    .withColumn(new_name, (regexp_replace(col(new_name),'[$,]','' )).cast("long"))
  elif new_name in ["artist", "tour_title"]:
    df = df.withColumn(new_name, (regexp_replace(col(new_name),r'\[.*?]','' ))) \
    .withColumn(new_name, (regexp_replace(col(new_name),r'[†‡\*]','' )).cast("string"))
  elif new_name == "year":
    df = df.withColumn("year_clean",regexp_replace(col("year"), "[–—]", "-")) \
    .withColumn("year_array",split(col("year_clean"), "-")) \
    .withColumn("start_year",get(col("year_array"), 0).cast("int")) \
    .withColumn("end_year",coalesce(get(col("year_array"), 1),get(col("year_array"), 0)).cast("int")) \
    .drop("year_clean", "year_array")
  elif new_name == "reference":
    df = df.withColumn("reference",regexp_replace(col("reference"),))

#year positioning
idx = df.columns.index("year")
columns = df.columns
columns.remove("year")
columns.remove("start_year")
columns.remove("end_year")
columns.insert(idx, "start_year")
columns.insert(idx+1, "end_year")
df = df.select(*columns)



In [92]:
# df = df.fillna(0,subset=["peak","all_time_peak"])
df.show()

+----+----+-------------+------------+--------------+------------+--------------------+----------+--------+-----+-------------+---------+
|rank|peak|all_time_peak|actual_gross|adjusted_gross|      artist|          tour_title|start_year|end_year|shows|average_gross|reference|
+----+----+-------------+------------+--------------+------------+--------------------+----------+--------+-----+-------------+---------+
|   1|   1|            2|   780000000|     780000000|Taylor Swift|      The Eras Tour |      2023|    2024|   56|     13928571|      [1]|
|   2|   1|            7|   579800000|     579800000|     Beyoncé|Renaissance World...|      2023|    2023|   56|     10353571|      [3]|
|   3|   1|            2|   411000000|     560622615|     Madonna|Sticky & Sweet Tour |      2008|    2009|   85|      4835294|      [6]|
|   4|   2|           10|   397300000|     454751555|        Pink|Beautiful Trauma ...|      2018|    2019|  156|      2546795|      [7]|
|   5|   2|         NULL|   345675